# Outlier-Robust Kernel Learning

Experiments comparing **Linear ITGD** and **Kernel ITGD** under the strong $\epsilon$-contamination model.

**Methods:**
- Linear ITGD — primal-space robust regression (Algorithm 1, Rathnashyam & Gittens)
- Kernel ITGD — dual-space extension with various kernels (Scheer)
- Huber KRR — kernel ridge regression with Huber loss (robust baseline: RKHS regularization + bounded-influence loss, but no filtering/thresholding)
- OLS baselines (corrupted / oracle inliers)

**Sections:**
1. Convergence check (linear data)
2. Method comparison table
3. $\epsilon$-sweep (robustness across corruption levels)
4. Nonlinear corrupted data — show that kernel methods work

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from itgd_2 import (
    itgd_linear, itgd_kernel, huber_krr,
    predict_linear, predict_kernel,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

ModuleNotFoundError: No module named 'itgd'

## Data generators

In [ ]:
def make_linear_data(n, d, w_star, sigma, eps, scale=50.0, seed=42):
    """y = x^T w* + noise, then corrupt eps*n samples (covariates + labels)."""
    rng = np.random.RandomState(seed)
    X = rng.randn(n, d)
    y = X @ w_star + sigma * rng.randn(n)
    X_c, y_c = X.copy(), y.copy()
    m = int(eps * n)
    idx = rng.choice(n, m, replace=False)
    X_c[idx] += scale * rng.randn(m, d)
    y_c[idx]  = scale * rng.randn(m)
    mask = np.zeros(n, bool); mask[idx] = True
    return X_c, y_c, X, y, mask


def make_nonlinear_data(n, d, w_star, sigma, eps, activation='leaky_relu',
                        scale=50.0, seed=42):
    """y = f(x^T w*) + noise, then corrupt eps*n samples."""
    rng = np.random.RandomState(seed)
    X = rng.randn(n, d)
    z = X @ w_star
    if activation == 'leaky_relu':
        f = np.where(z >= 0, z, 0.1 * z)
    elif activation == 'relu':
        f = np.maximum(z, 0)
    else:
        f = z
    y = f + sigma * rng.randn(n)
    X_c, y_c = X.copy(), y.copy()
    m = int(eps * n)
    idx = rng.choice(n, m, replace=False)
    X_c[idx] += scale * rng.randn(m, d)
    y_c[idx]  = scale * rng.randn(m)
    mask = np.zeros(n, bool); mask[idx] = True
    return X_c, y_c, X, y, mask

---
## 1. Convergence check (linear data, $\epsilon = 0.15$)

In [ ]:
n, d, sigma, eps = 800, 20, 1.0, 0.15
rng = np.random.RandomState(0)
w_star = rng.randn(d); w_star *= 3.0 / np.linalg.norm(w_star)
Sigma = np.eye(d)

X_c, y_c, X, y, corrupted = make_linear_data(n, d, w_star, sigma, eps, seed=123)
idx_tr, idx_te = train_test_split(np.arange(n), test_size=0.3, random_state=42)
X_tr, y_tr = X_c[idx_tr], y_c[idx_tr]
X_te, y_te = X[idx_te],  y[idx_te]

print(f'Train: {len(idx_tr)}  Test: {len(idx_te)}  Corrupted in train: {corrupted[idx_tr].sum()}')

In [ ]:
T = 500

w_hat, h_lin = itgd_linear(X_tr, y_tr, Sigma, eps, eta=0.1, T=T)
a_kl, Xw_kl, h_kl = itgd_kernel(X_tr, y_tr, Sigma, eps, eta=0.01,
                                  kernel='linear', lam=1e-3, T=T)
a_kr, Xw_kr, h_kr = itgd_kernel(X_tr, y_tr, Sigma, eps, eta=0.005,
                                  kernel='rbf', lam=1e-3, T=T, gamma=0.5/d)
a_kp, Xw_kp, h_kp = itgd_kernel(X_tr, y_tr, Sigma, eps, eta=0.01,
                                  kernel='poly', lam=1e-3, T=T, degree=2)

# Huber KRR baselines (same kernels, no filtering/thresholding)
a_hl, Xw_hl, h_hl = huber_krr(X_tr, y_tr, Sigma, kernel='linear',
                                lam=1e-3, delta=1.0, eta=0.01, T=T)
a_hr, Xw_hr, h_hr = huber_krr(X_tr, y_tr, Sigma, kernel='rbf',
                                lam=1e-3, delta=1.0, eta=0.005, T=T, gamma=0.5/d)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for label, h, c, ls in [('Linear ITGD',     h_lin, 'C0', '-'),
                         ('Kernel (linear)', h_kl,  'C1', '-'),
                         ('Kernel (RBF)',    h_kr,  'C2', '-'),
                         ('Kernel (poly-2)', h_kp,  'C3', '-'),
                         ('Huber KRR (lin)', h_hl,  'C1', '--'),
                         ('Huber KRR (RBF)', h_hr,  'C2', '--')]:
    axes[0].semilogy(h['grad_norm'], ls, label=label, color=c, alpha=.8)
    axes[1].semilogy(h['mse_full'],  ls, label=label, color=c, alpha=.8)

axes[0].set(xlabel='Iteration', ylabel='Gradient norm', title='Gradient convergence')
axes[1].set(xlabel='Iteration', ylabel='MSE (all train pts)', title='Training MSE')
for ax in axes: ax.legend(fontsize=7); ax.grid(True, alpha=.3)
plt.tight_layout(); plt.show()

---
## 2. Method comparison (Test MSE)

In [ ]:
mse = lambda p: float(np.mean((p - y_te)**2))

results = {
    'Linear ITGD':        mse(predict_linear(X_te, w_hat)),
    'Kernel ITGD (lin)':  mse(predict_kernel(X_te, Xw_kl, a_kl, Sigma, 'linear')),
    'Kernel ITGD (RBF)':  mse(predict_kernel(X_te, Xw_kr, a_kr, Sigma, 'rbf', gamma=0.5/d)),
    'Kernel ITGD (poly)': mse(predict_kernel(X_te, Xw_kp, a_kp, Sigma, 'poly', degree=2)),
    'Huber KRR (lin)':    mse(predict_kernel(X_te, Xw_hl, a_hl, Sigma, 'linear')),
    'Huber KRR (RBF)':    mse(predict_kernel(X_te, Xw_hr, a_hr, Sigma, 'rbf', gamma=0.5/d)),
    'OLS (corrupted)':    mse(X_te @ np.linalg.lstsq(X_tr, y_tr, rcond=None)[0]),
    'OLS (inliers)':      mse(X_te @ np.linalg.lstsq(X_tr[~corrupted[idx_tr]],
                                                       y_tr[~corrupted[idx_tr]], rcond=None)[0]),
}

print(f'{"Method":<22} {"Test MSE":>10}')
print('-' * 34)
for k, v in results.items():
    print(f'{k:<22} {v:10.4f}')

---
## 3. Epsilon sweep — robustness across corruption levels

In [ ]:
epsilons = [0.02, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
sweep = {k: [] for k in ['Lin ITGD', 'K-Linear', 'K-RBF',
                          'Huber-Lin', 'Huber-RBF',
                          'OLS (corr)', 'OLS (inlier)']}

for ep in epsilons:
    Xc, yc, Xcl, ycl, mask = make_linear_data(n, d, w_star, sigma, ep, seed=42)
    itr, ite = train_test_split(np.arange(n), test_size=.3, random_state=42)
    Xt, yt = Xc[itr], yc[itr]
    Xv, yv = Xcl[ite], ycl[ite]
    mt = mask[itr]
    mse_ = lambda p: float(np.mean((p - yv)**2))

    w, _ = itgd_linear(Xt, yt, Sigma, ep, eta=.1, T=T)
    sweep['Lin ITGD'].append(mse_(predict_linear(Xv, w)))

    a, Xw, _ = itgd_kernel(Xt, yt, Sigma, ep, eta=.01, kernel='linear', lam=1e-3, T=T)
    sweep['K-Linear'].append(mse_(predict_kernel(Xv, Xw, a, Sigma, 'linear')))

    a, Xw, _ = itgd_kernel(Xt, yt, Sigma, ep, eta=.005, kernel='rbf', lam=1e-3, T=T, gamma=.5/d)
    sweep['K-RBF'].append(mse_(predict_kernel(Xv, Xw, a, Sigma, 'rbf', gamma=.5/d)))

    # Huber KRR baselines (no epsilon arg — robustness from loss, not filtering)
    a, Xw, _ = huber_krr(Xt, yt, Sigma, kernel='linear', lam=1e-3, delta=1.0, eta=.01, T=T)
    sweep['Huber-Lin'].append(mse_(predict_kernel(Xv, Xw, a, Sigma, 'linear')))

    a, Xw, _ = huber_krr(Xt, yt, Sigma, kernel='rbf', lam=1e-3, delta=1.0, eta=.005, T=T, gamma=.5/d)
    sweep['Huber-RBF'].append(mse_(predict_kernel(Xv, Xw, a, Sigma, 'rbf', gamma=.5/d)))

    sweep['OLS (corr)'].append(mse_(Xv @ np.linalg.lstsq(Xt, yt, rcond=None)[0]))
    sweep['OLS (inlier)'].append(mse_(Xv @ np.linalg.lstsq(Xt[~mt], yt[~mt], rcond=None)[0]))

    print(f'eps={ep:.2f}  Lin={sweep["Lin ITGD"][-1]:.3f}  '
          f'K-Lin={sweep["K-Linear"][-1]:.3f}  K-RBF={sweep["K-RBF"][-1]:.3f}  '
          f'Hub-L={sweep["Huber-Lin"][-1]:.3f}  Hub-R={sweep["Huber-RBF"][-1]:.3f}  '
          f'OLS={sweep["OLS (corr)"][-1]:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
styles = {'Lin ITGD':    ('C0','o-'),
          'K-Linear':    ('C1','s-'),
          'K-RBF':       ('C2','^-'),
          'Huber-Lin':   ('C1','s--'),
          'Huber-RBF':   ('C2','^--'),
          'OLS (corr)':  ('C3','x--'),
          'OLS (inlier)':('gray','*--')}
for k, (c, fmt) in styles.items():
    ax.plot(epsilons, sweep[k], fmt, color=c, label=k, markersize=5)
ax.set(xlabel='Corruption fraction $\\epsilon$', ylabel='Test MSE (log)',
       title='Robustness: Test MSE vs corruption level')
ax.set_yscale('log'); ax.legend(fontsize=8); ax.grid(True, alpha=.3)
plt.tight_layout(); plt.show()

---
## 4. Nonlinear data — when does the kernel method help?

Data is generated as $y = \text{LeakyReLU}(\mathbf{x}^\top \mathbf{w}_*)+ \xi$ then corrupted.

We compare:
- **Linear ITGD** with identity activation (misspecified model)
- **Linear ITGD** with leaky-ReLU activation (correctly specified)
- **Kernel ITGD (RBF, identity)** — the RBF kernel can capture the nonlinearity implicitly
- **Kernel ITGD (RBF, leaky-ReLU)** — both kernel *and* activation
- **Huber KRR (RBF, identity)** — same RBF kernel as ITGD but robustness comes only from the Huber loss (no filtering/thresholding). Isolates how much of the robustness is due to filtering + thresholding.

In [ ]:
n_nl, d_nl, sigma_nl = 1000, 15, 0.5
rng_nl = np.random.RandomState(7)
w_star_nl = rng_nl.randn(d_nl); w_star_nl *= 2.0 / np.linalg.norm(w_star_nl)
Sigma_nl = np.eye(d_nl)

eps_list = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
nl_results = {k: [] for k in [
    'Lin ITGD (identity)', 'Lin ITGD (leaky)',
    'K-RBF (identity)', 'K-RBF (leaky)',
    'Huber-RBF (identity)',
    'OLS (corr)', 'OLS (inlier)'
]}

T_nl = 600
for ep in eps_list:
    Xc, yc, Xcl, ycl, mask = make_nonlinear_data(
        n_nl, d_nl, w_star_nl, sigma_nl, ep, 'leaky_relu', seed=99)
    itr, ite = train_test_split(np.arange(n_nl), test_size=.3, random_state=42)
    Xt, yt, mt = Xc[itr], yc[itr], mask[itr]
    Xv, yv = Xcl[ite], ycl[ite]
    mse_ = lambda p: float(np.mean((p - yv)**2))

    # Linear ITGD, identity (misspecified)
    w, _ = itgd_linear(Xt, yt, Sigma_nl, ep, eta=.1, T=T_nl, activation='identity')
    nl_results['Lin ITGD (identity)'].append(mse_(predict_linear(Xv, w, 'identity')))

    # Linear ITGD, leaky relu
    w, _ = itgd_linear(Xt, yt, Sigma_nl, ep, eta=.1, T=T_nl, activation='leaky_relu')
    nl_results['Lin ITGD (leaky)'].append(mse_(predict_linear(Xv, w, 'leaky_relu')))

    # Kernel RBF, identity — kernel absorbs nonlinearity
    a, Xw, _ = itgd_kernel(Xt, yt, Sigma_nl, ep, eta=.005, kernel='rbf',
                            lam=1e-3, T=T_nl, activation='identity', gamma=.5/d_nl)
    nl_results['K-RBF (identity)'].append(
        mse_(predict_kernel(Xv, Xw, a, Sigma_nl, 'rbf', 'identity', gamma=.5/d_nl)))

    # Kernel RBF, leaky relu
    a, Xw, _ = itgd_kernel(Xt, yt, Sigma_nl, ep, eta=.005, kernel='rbf',
                            lam=1e-3, T=T_nl, activation='leaky_relu', gamma=.5/d_nl)
    nl_results['K-RBF (leaky)'].append(
        mse_(predict_kernel(Xv, Xw, a, Sigma_nl, 'rbf', 'leaky_relu', gamma=.5/d_nl)))

    # Huber KRR RBF, identity — no filtering/thresholding baseline
    a, Xw, _ = huber_krr(Xt, yt, Sigma_nl, kernel='rbf', lam=1e-3, delta=1.0,
                          eta=.005, T=T_nl, activation='identity', gamma=.5/d_nl)
    nl_results['Huber-RBF (identity)'].append(
        mse_(predict_kernel(Xv, Xw, a, Sigma_nl, 'rbf', 'identity', gamma=.5/d_nl)))

    # OLS baselines (leaky-relu applied at prediction)
    w_ols = np.linalg.lstsq(Xt, yt, rcond=None)[0]
    nl_results['OLS (corr)'].append(mse_(predict_linear(Xv, w_ols, 'leaky_relu')))
    w_cl = np.linalg.lstsq(Xt[~mt], yt[~mt], rcond=None)[0]
    nl_results['OLS (inlier)'].append(mse_(predict_linear(Xv, w_cl, 'leaky_relu')))

    print(f'eps={ep:.2f}  Lin-id={nl_results["Lin ITGD (identity)"][-1]:.3f}  '
          f'Lin-lk={nl_results["Lin ITGD (leaky)"][-1]:.3f}  '
          f'RBF-id={nl_results["K-RBF (identity)"][-1]:.3f}  '
          f'RBF-lk={nl_results["K-RBF (leaky)"][-1]:.3f}  '
          f'Hub-RBF={nl_results["Huber-RBF (identity)"][-1]:.3f}  '
          f'OLS={nl_results["OLS (corr)"][-1]:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
styles_nl = {
    'Lin ITGD (identity)':  ('C0', 'o--'),
    'Lin ITGD (leaky)':     ('C0', 'o-'),
    'K-RBF (identity)':     ('C2', '^--'),
    'K-RBF (leaky)':        ('C2', '^-'),
    'Huber-RBF (identity)': ('C4', 'd--'),
    'OLS (corr)':           ('C3', 'x--'),
    'OLS (inlier)':         ('gray', '*--'),
}
for k, (c, fmt) in styles_nl.items():
    ax.plot(eps_list, nl_results[k], fmt, color=c, label=k, markersize=5)
ax.set(xlabel='Corruption fraction $\\epsilon$', ylabel='Test MSE (log)',
       title='Nonlinear (Leaky-ReLU) data — Test MSE vs corruption')
ax.set_yscale('log'); ax.legend(fontsize=7); ax.grid(True, alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
print(f'{"Method":<24}', '  '.join(f'eps={e:.2f}' for e in eps_list))
print('-' * (24 + 10 * len(eps_list)))
for k in nl_results:
    vals = '  '.join(f'{v:7.4f}' for v in nl_results[k])
    print(f'{k:<24} {vals}')